# House Price Prediction — 02: Data Preprocessing

This notebook applies the decisions made during EDA (`01_eda.ipynb`) and prepares the data for modeling.

**Decisions carried over from EDA:**
1. Drop rows where `AveOccup > 10` (institutional/group housing artifact)
2. Drop rows where `AveRooms > 20` (tiny-household-count artifact — also resolves `AveBedrms` outliers)
3. Keep `Population`'s wide range and `MedHouseVal`'s capping at 5.0 — both are legitimate/documented, not removed
4. Engineer `is_max_house_age` flag (`HouseAge == 52` is a censored/capped value, not a literal precise age)
5. Engineer `bedroom_ratio = AveBedrms / AveRooms`, then drop `AveBedrms` (was 0.85 correlated with `AveRooms`)
6. No geographic distance-to-city features — relying on tree-based models to capture lat/long interactions naturally
7. Log-transform candidates (`MedInc`, `Population`) kept as *additional* columns, not replacements — useful for linear models, not needed for tree-based models. The choice of which to use happens in `03_model_training.ipynb`, per model.

**Steps in this notebook:**
- Load data, apply outlier mask
- Feature engineering
- Stratified train/test split (stratified on income, since `MedInc` is the strongest predictor — same rationale as census-based sampling, ensures train/test sets are representative across income levels)
- Scaling (fit on training data only — no leakage)
- Save processed splits + scaler to `outputs/`

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

os.makedirs('outputs', exist_ok=True)
print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load Data

In [2]:
df = pd.read_csv('housing.csv')
print(f'Initial shape: {df.shape}')
df.head()

Initial shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 3. Outlier Removal

See `01_eda.ipynb` section on `AveOccup`/`AveRooms`/`AveBedrms` investigation for full reasoning.

In [3]:
mask = (df['AveOccup'] <= 10) & (df['AveRooms'] <= 20)

df_clean = df[mask].copy()

rows_removed = len(df) - len(df_clean)
print(f"Rows removed: {rows_removed} ({rows_removed/len(df)*100:.2f}%)")
print(f"Remaining rows: {len(df_clean)}")

df_clean[['AveRooms', 'AveBedrms', 'Population', 'AveOccup']].describe()

Rows removed: 105 (0.51%)
Remaining rows: 20535


,AveRooms,AveBedrms,Population,AveOccup
count,20535.000000,20535.000000,20535.000000,20535.000000
mean,5.335105,1.077498,1427.048454,2.920374
std,1.420036,0.197741,1128.128629,0.765425
min,0.846154,0.333333,3.000000,0.750000
25%,4.438131,1.005908,791.000000,2.431048
50%,5.225127,1.048409,1168.000000,2.818966
75%,6.041772,1.098592,1726.000000,3.280652
max,19.962121,6.500000,35682.000000,9.954545


## 4. Feature Engineering

### 4.1 `is_max_house_age` flag

`HouseAge` is capped at 52 (top-coded in the original census data). This flag lets the model distinguish "genuinely 52 years old" from "52-or-older, unknown," rather than treating the cap as a literal precise value.

In [4]:
df_clean['is_max_house_age'] = (df_clean['HouseAge'] == 52).astype(int)

print(df_clean['is_max_house_age'].value_counts())

is_max_house_age
0    19269
1     1266
Name: count, dtype: int64


### 4.2 `bedroom_ratio` (replaces `AveBedrms`)

`AveRooms` and `AveBedrms` were correlated at 0.85. Rather than dropping one outright, the ratio captures the *layout/type* signal (bedroom-heavy vs. spacious) while `AveRooms` is kept to preserve the *size* signal.

In [5]:
df_clean['bedroom_ratio'] = df_clean['AveBedrms'] / df_clean['AveRooms']

df_clean = df_clean.drop(columns=['AveBedrms'])

df_clean[['AveRooms', 'bedroom_ratio']].describe()

,AveRooms,bedroom_ratio
count,20535.000000,20535.000000
mean,5.335105,0.213050
std,1.420036,0.057769
min,0.846154,0.100000
25%,4.438131,0.175433
50%,5.225127,0.203187
75%,6.041772,0.239849
max,19.962121,1.000000


In [6]:
# Confirm the redundancy was actually resolved, and check if bedroom_ratio
# shows any correlation with the target that AveBedrms alone didn't.
df_clean.corr(numeric_only=True)['MedHouseVal'].sort_values(ascending=False)

MedHouseVal         1.000000
MedInc              0.689668
AveRooms            0.274243
is_max_house_age    0.151899
HouseAge            0.104838
Population         -0.025343
Longitude          -0.045664
Latitude           -0.144098
bedroom_ratio      -0.256867
AveOccup           -0.267194
Name: MedHouseVal, dtype: float64

### 4.3 Log-transform candidates (kept as additional columns)

`MedInc` and `Population` are right-skewed. Adding `log1p` versions as *extra* columns (not replacements) keeps the option open for linear models in `03_model_training.ipynb`, without forcing the transform on tree-based models that don't need it.

In [7]:
df_clean['log_MedInc'] = np.log1p(df_clean['MedInc'])
df_clean['log_Population'] = np.log1p(df_clean['Population'])

df_clean[['MedInc', 'log_MedInc', 'Population', 'log_Population']].describe()

,MedInc,log_MedInc,Population,log_Population
count,20535.000000,20535.000000,20535.000000,20535.000000
mean,3.871385,1.517331,1427.048454,7.031068
std,1.896434,0.358182,1128.128629,0.725573
min,0.499900,0.405398,3.000000,1.386294
25%,2.565150,1.271206,791.000000,6.674561
50%,3.535700,1.511979,1168.000000,7.063904
75%,4.745650,1.748443,1726.000000,7.454141
max,15.000100,2.772595,35682.000000,10.482430


## 5. Final Feature Check

Confirm the full set of columns before splitting — make sure nothing unexpected slipped through.

In [8]:
print(f"Shape after feature engineering: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")
df_clean.head()

Shape after feature engineering: (20535, 12)
Columns: ['MedInc', 'HouseAge', 'AveRooms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal', 'is_max_house_age', 'bedroom_ratio', 'log_MedInc', 'log_Population']


,MedInc,HouseAge,AveRooms,Population,AveOccup,Latitude,Longitude,MedHouseVal,is_max_house_age,bedroom_ratio,log_MedInc,log_Population
0,8.3252,41.0,6.984127,322.0,2.555556,37.88,-122.23,4.526,0,0.146591,2.232720,5.777652
1,8.3014,21.0,6.238137,2401.0,2.109842,37.86,-122.22,3.585,0,0.155797,2.230165,7.784057
2,7.2574,52.0,8.288136,496.0,2.802260,37.85,-122.24,3.521,1,0.129516,2.111110,6.208590
3,5.6431,52.0,5.817352,558.0,2.547945,37.85,-122.25,3.413,1,0.184458,1.893579,6.326149
4,3.8462,52.0,6.281853,565.0,2.181467,37.85,-122.25,3.422,1,0.172096,1.578195,6.338594


## 6. Stratified Train/Test Split

Since `MedInc` is by far the strongest predictor (0.69 correlation with the target), a plain random split risks the train and test sets having meaningfully different income distributions just by chance. Binning `MedInc` into categories and stratifying on those bins ensures both sets are representative across the income spectrum — the same principle used for `Churn`-stratified splitting in the churn project, just applied to a continuous feature here instead of the target.

In [9]:
df_clean['income_cat'] = pd.cut(
    df_clean['MedInc'],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

df_clean['income_cat'].value_counts().sort_index()

income_cat
1     813
2    6550
3    7193
4    3626
5    2353
Name: count, dtype: int64

In [10]:
X = df_clean.drop(columns=['MedHouseVal', 'income_cat'])
y = df_clean['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['income_cat']
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (16428, 11)
Test shape: (4107, 11)


In [11]:
# Sanity check: confirm the stratification actually worked —
# income category proportions should look similar in train vs. test
train_props = df_clean.loc[X_train.index, 'income_cat'].value_counts(normalize=True).sort_index()
test_props = df_clean.loc[X_test.index, 'income_cat'].value_counts(normalize=True).sort_index()

pd.DataFrame({'Train %': train_props, 'Test %': test_props}).round(3)

,Train %,Test %
income_cat,,
1,0.040,0.040
2,0.319,0.319
3,0.350,0.350
4,0.177,0.177
5,0.115,0.115


## 7. Scaling

Fit `StandardScaler` on training data only — the same leakage-prevention principle from the churn project. The scaler learns mean/std from `X_train` exclusively, then applies that same transformation to `X_test`.

Note: scaling matters for linear models and distance-based models, but tree-based models (Random Forest, Gradient Boosting) are unaffected by it. Both scaled and unscaled versions are saved so `03_model_training.ipynb` can pick whichever a given model needs.

In [12]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled.describe().T[['mean', 'std']]

,mean,std
MedInc,7.785348e-17,1.00003
HouseAge,3.287147e-17,1.00003
AveRooms,-2.655669e-16,1.00003
Population,4.973972e-17,1.00003
AveOccup,9.515425e-18,1.00003
Latitude,-5.017224e-17,1.00003
Longitude,3.264656e-15,1.00003
is_max_house_age,-2.811376e-17,1.00003
bedroom_ratio,-4.973972e-18,1.00003
log_MedInc,4.026755e-16,1.00003


## 8. Save Outputs

Saving both unscaled (for tree-based models) and scaled (for linear models) train/test splits, plus the fitted scaler itself, so it can be reused on new data at inference time.

In [13]:
artifacts = {
    'X_train': X_train,
    'X_test': X_test,
    'X_train_scaled': X_train_scaled,
    'X_test_scaled': X_test_scaled,
    'y_train': y_train,
    'y_test': y_test,
}

for name, obj in artifacts.items():
    with open(f'outputs/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)

with open('outputs/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Saved to outputs/:')
for name in list(artifacts.keys()) + ['scaler']:
    print(f'  - {name}.pkl')

Saved to outputs/:
  - X_train.pkl
  - X_test.pkl
  - X_train_scaled.pkl
  - X_test_scaled.pkl
  - y_train.pkl
  - y_test.pkl
  - scaler.pkl


## 9. Summary

- **Rows removed:** outlier mask (`AveOccup <= 10` & `AveRooms <= 20`)
- **Features engineered:** `is_max_house_age`, `bedroom_ratio` (dropped `AveBedrms`), `log_MedInc`, `log_Population` (kept alongside originals)
- **Split:** 80/20 train/test, stratified by binned `MedInc` (`income_cat`) to keep income distribution representative in both sets
- **Scaling:** `StandardScaler` fit on training data only; both scaled and unscaled versions saved
- **Known limitation carried forward:** `MedHouseVal` is capped at 5.00001 (~$500,001) for ~4.8% of rows — model will likely underpredict for genuinely high-value properties

**Next notebook:** `03_model_training.ipynb` — train and compare multiple regression models (e.g. Linear Regression, Ridge/Lasso, Random Forest, Gradient Boosting/XGBoost), decide which feature set (scaled vs. unscaled, log-transformed vs. raw) suits each, and select a champion based on appropriate regression metrics (RMSE, MAE, R²).